In [ ]:
from andromeda.core.agent import Agent
from andromeda.core.workflow import WorkflowBuilder
from andromeda.config import AgentConfig, ModelConfig
from andromeda import HumanMessage
from pydantic import BaseModel
from typing import Optional
from pprint import pprint

In [18]:
class AgentState(BaseModel):
    query : str
    classification : Optional[str]
    answer : Optional[str]
    humanreview : Optional[str]

In [19]:
model_config = ModelConfig(name = 'llama3.2:3b', provider = 'ollama', temperature = 0.5)

In [20]:
classiferAgent = Agent(
    AgentConfig(
        name = 'classifier_agent',
        model = model_config,
        prompt = 'Reply with exactly one word: SIMPLE or COMPLEX  , classifying the users request. No extra expkanation'
    )
)

In [21]:
deep_researcher = Agent(
    AgentConfig(
        name = 'deep_researcher_agent',
        model = model_config,
        prompt = 'Do thorough multi-angle research and give a detialed answer'
    )
)

In [22]:
quick_researcher = Agent(
    AgentConfig(
        name = 'quick_researcher_agent',
        model = model_config,
        prompt = 'Do thorough multi-angle research and give a detialed answer'
    )
)

In [23]:
def classify_query(state:AgentState):
    reply = classiferAgent.invoke([HumanMessage(content=state['query'])])
    label = reply[-1].content.strip().upper()
    if "COMPLEX" in label:
        raise RuntimeError("complex")
    return {"classification":"SIMPLE"}

In [24]:
def run_deep_research(state:AgentState):
    reply = deep_researcher.invoke([HumanMessage(content = state['query'])])
    response = reply[-1].content
    return {
        'classification' : "COMPLEX",
        'answer' : respsone
    }

In [ ]:
def run_simple_research(state:AgentState):
    reply = quick_researcher.invoke([HumanMessage(content = state['query'])])
    response = reply[-1].content
    return {
        'classification' : "SIMPLE",
        'answer' : response
    }

In [26]:
pipeline = WorkflowBuilder(name = "Branching_HITL_Research_Pipeline")
(
    pipeline
    .start('classify').run(classify_query)
        .if_fails().goto('deep_research')
        .if_succeeds().goto('quick_answer')
    .then('deep_research').run(run_deep_research)
    .then('quick_answer').run(run_simple_research)

)

In [27]:
response = pipeline.execute(state = {"query":"Compare RAFT and Raft-lite consensus tradeoffs"})

WorkflowExecutionError: name 'respsone' is not defined

In [ ]:
pprint(response)